In [0]:
from pyspark.sql.functions import avg, year, trim

# Filter yield data to desired states (remove leading spaces as necessary)
states = ['Nebraska','Kansas','South Dakota','Missouri','Iowa','Indiana','Illinois','Ohio','Kentucky']
df_yield = spark.sql("""
    SELECT * FROM workspace.default.rma_county_yields_report_399
    WHERE trim(`State Name`) IN ({})
""".format(','.join([f"'{s}'" for s in states])))

# Aggregate weather data by year
df_weather = spark.sql("""
    SELECT station, latitude, longitude, date, precipitation, temp_max, temp_min
    FROM rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily
""")
df_weather = df_weather.withColumn('weather_year', year(df_weather['date']))
df_weather_agg = df_weather.groupBy('weather_year').agg(
    avg('precipitation').alias('avg_precipitation'),
    avg('temp_max').alias('avg_temp_max'),
    avg('temp_min').alias('avg_temp_min')
)

# Join yield and weather data
df_joined = df_yield.join(df_weather_agg, df_yield['Yield Year'] == df_weather_agg['weather_year'], 'inner')

# Select modeling features
df_model = df_joined.select(
    'Commodity Name','State Name','County Name','Irrigation Practice Name',
    'Yield Year','Yield Amount','avg_precipitation','avg_temp_max','avg_temp_min'
)

In [0]:
pdf = df_model.toPandas()

In [0]:
from sklearn.preprocessing import OneHotEncoder

categorical_cols = ['Commodity Name','State Name','County Name','Irrigation Practice Name']
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = encoder.fit_transform(pdf[categorical_cols])
import pandas as pd
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_cols))

numerical_cols = ['Yield Year','avg_precipitation','avg_temp_max','avg_temp_min']
X = pd.concat([encoded_df, pdf[numerical_cols]], axis=1)
y = pdf['Yield Amount']

In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2:", r2)
feature_imp = pd.Series(rf.feature_importances_, index=X.columns)
print("Top Feature Importances:")
print(feature_imp.sort_values(ascending=False).head(10))

In [0]:
# Simulate 30% less precipitation for scenario yield
X_scenario = X_test.copy()
X_scenario['avg_precipitation'] = X_scenario['avg_precipitation'] * 0.7
y_scenario_pred = rf.predict(X_scenario)

scenario_yield_change = (y_scenario_pred.mean() - y_pred.mean()) / y_pred.mean()
print("Scenario Yield Change (%):", scenario_yield_change * 100)

In [0]:
pd.DataFrame({'Actual': y_test.values[:5], 'Predicted': y_pred[:5]})